# Proyecto de Grado: Priorizacion de Mantenimiento Predictivo para Fallas APS en Camiones Scania

Este notebook contiene el flujo principal del proyecto.

Objetivo:
- desarrollar un pipeline reproducible para detectar riesgo de falla en el sistema APS
- comparar modelos base y avanzados
- optimizar la decision con costo de error y seleccion de umbral
- generar una salida de priorizacion util para una capa de aplicacion posterior

Salidas del notebook:
- tablas y figuras para el informe
- modelo seleccionado
- umbral recomendado
- tabla de priorizacion sobre el conjunto de prueba oficial


## 1. Configuracion e importaciones

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
REPORTS_DIR = PROJECT_ROOT / 'reports' / 'scania_project'
FIGURES_DIR = REPORTS_DIR / 'figures'
TABLES_DIR = REPORTS_DIR / 'tables'
MODELS_DIR = PROJECT_ROOT / 'models'

for directory in [REPORTS_DIR, FIGURES_DIR, TABLES_DIR, MODELS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = RAW_DIR / 'aps_failure_training_set.csv'
TEST_PATH = RAW_DIR / 'aps_failure_test_set.csv'
TARGET_COLUMN = 'class'
POSITIVE_LABEL = 'pos'
RANDOM_STATE = 42
FALSE_POSITIVE_COST = 10
FALSE_NEGATIVE_COST = 500

TRAIN_PATH, TEST_PATH

## 2. Funciones auxiliares

In [ ]:
def load_scania_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, na_values='na', skiprows=19, low_memory=False)
    df.columns = df.columns.str.strip()
    return df


def cost_from_confusion(cm, fp_cost=FALSE_POSITIVE_COST, fn_cost=FALSE_NEGATIVE_COST):
    tn, fp, fn, tp = cm.ravel()
    return fp * fp_cost + fn * fn_cost


def summarize_predictions(y_true, predictions, positive_label=POSITIVE_LABEL):
    cm = confusion_matrix(y_true, predictions, labels=['neg', positive_label])
    return {
        'accuracy': accuracy_score(y_true, predictions),
        'f1_macro': f1_score(y_true, predictions, average='macro'),
        'precision_pos': precision_score(y_true, predictions, pos_label=positive_label, zero_division=0),
        'recall_pos': recall_score(y_true, predictions, pos_label=positive_label, zero_division=0),
        'f1_pos': f1_score(y_true, predictions, pos_label=positive_label, zero_division=0),
        'cm': cm,
        'cost': cost_from_confusion(cm),
    }


def save_dataframe(df: pd.DataFrame, filename: str):
    path = TABLES_DIR / filename
    df.to_csv(path, index=False)
    return path


## 3. Carga y caracterizacion de datos

In [ ]:
train_df = load_scania_csv(TRAIN_PATH)
test_df = load_scania_csv(TEST_PATH)

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)

train_df.head()

In [ ]:
missing_top15 = train_df.isna().mean().sort_values(ascending=False).head(15).reset_index()
missing_top15.columns = ['feature', 'missing_ratio']
save_dataframe(missing_top15, 'missing_top15.csv')
missing_top15

In [ ]:
class_distribution = train_df[TARGET_COLUMN].value_counts().rename_axis('class').reset_index(name='count')
class_distribution['proportion'] = class_distribution['count'] / class_distribution['count'].sum()
save_dataframe(class_distribution, 'class_distribution.csv')
class_distribution

**Conclusión de la caracterización:** El dataset tiene un tamaño suficiente para modelado (`60000` registros de entrenamiento), un desbalance fuerte en la clase positiva y un volumen relevante de valores faltantes. Esto justifica el uso de métricas distintas a `accuracy`, técnicas robustas frente a faltantes y una formulación del problema orientada a priorización de riesgo.

## 4. Preparacion de entrenamiento y validacion interna

In [ ]:
X = train_df.drop(columns=[TARGET_COLUMN]).copy()
y = train_df[TARGET_COLUMN].copy()

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train.shape, X_valid.shape

## 5. Modelos de referencia

In [ ]:
dummy_model = DummyClassifier(strategy='most_frequent')
dummy_model.fit(X_train, y_train)
dummy_predictions = dummy_model.predict(X_valid)
dummy_summary = summarize_predictions(y_valid, dummy_predictions)
dummy_summary

In [ ]:
logistic_model = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced')),
    ]
)
logistic_model.fit(X_train, y_train)
logistic_predictions = logistic_model.predict(X_valid)
logistic_scores = logistic_model.predict_proba(X_valid)[:, list(logistic_model.classes_).index(POSITIVE_LABEL)]
logistic_summary = summarize_predictions(y_valid, logistic_predictions)
logistic_summary

In [ ]:
catboost_model = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.08,
    loss_function='Logloss',
    eval_metric='PRAUC',
    auto_class_weights='Balanced',
    verbose=False,
    random_state=RANDOM_STATE,
)
catboost_model.fit(X_train, y_train)
catboost_predictions = pd.Series(catboost_model.predict(X_valid).flatten(), index=y_valid.index)
catboost_scores = catboost_model.predict_proba(X_valid)[:, list(catboost_model.classes_).index(POSITIVE_LABEL)]
catboost_summary = summarize_predictions(y_valid, catboost_predictions)
catboost_summary

## 6. Comparacion de modelos

In [ ]:
model_summary = pd.DataFrame([
    {
        'model': 'Dummy',
        'accuracy': dummy_summary['accuracy'],
        'f1_macro': dummy_summary['f1_macro'],
        'precision_pos': dummy_summary['precision_pos'],
        'recall_pos': dummy_summary['recall_pos'],
        'f1_pos': dummy_summary['f1_pos'],
        'avg_precision_pos': np.nan,
        'cost': dummy_summary['cost'],
    },
    {
        'model': 'Logistic Regression',
        'accuracy': logistic_summary['accuracy'],
        'f1_macro': logistic_summary['f1_macro'],
        'precision_pos': logistic_summary['precision_pos'],
        'recall_pos': logistic_summary['recall_pos'],
        'f1_pos': logistic_summary['f1_pos'],
        'avg_precision_pos': average_precision_score((y_valid == POSITIVE_LABEL).astype(int), logistic_scores),
        'cost': logistic_summary['cost'],
    },
    {
        'model': 'CatBoost',
        'accuracy': catboost_summary['accuracy'],
        'f1_macro': catboost_summary['f1_macro'],
        'precision_pos': catboost_summary['precision_pos'],
        'recall_pos': catboost_summary['recall_pos'],
        'f1_pos': catboost_summary['f1_pos'],
        'avg_precision_pos': average_precision_score((y_valid == POSITIVE_LABEL).astype(int), catboost_scores),
        'cost': catboost_summary['cost'],
    },
])

save_dataframe(model_summary, 'model_summary.csv')
model_summary.round(4)

**Conclusión de la comparación de modelos:** El baseline trivial confirma que el problema no puede evaluarse únicamente con `accuracy`. La regresión logística ya captura señal útil, pero `CatBoost` logra el mejor equilibrio entre `f1_pos`, `average precision` y costo, por lo que se selecciona como modelo principal del proyecto.

## 7. Curva precision-recall del modelo seleccionado

In [ ]:
y_valid_pos = (y_valid == POSITIVE_LABEL).astype(int)
logistic_precision, logistic_recall, _ = precision_recall_curve(y_valid_pos, logistic_scores)
catboost_precision, catboost_recall, _ = precision_recall_curve(y_valid_pos, catboost_scores)

plt.figure(figsize=(8, 5))
plt.plot(logistic_recall, logistic_precision, label='Logistic regression')
plt.plot(catboost_recall, catboost_precision, label='CatBoost')
plt.xlabel('Recall for pos')
plt.ylabel('Precision for pos')
plt.title('Precision-Recall Curve for APS Failure Detection')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'precision_recall_curve.png', dpi=300, bbox_inches='tight')
plt.show()

**Conclusión de la curva precision-recall:** La curva de `CatBoost` domina a la de la regresión logística en la mayor parte del rango, lo que indica una mejor capacidad para ordenar los casos según riesgo. Esto refuerza su uso como base para priorización y no solo como clasificador binario.

## 8. Seleccion de umbral por costo

In [ ]:
threshold_rows = []
for threshold in np.arange(0.10, 0.95, 0.05):
    preds = np.where(catboost_scores >= threshold, POSITIVE_LABEL, 'neg')
    summary = summarize_predictions(y_valid, preds)
    threshold_rows.append({
        'threshold': round(float(threshold), 2),
        'precision_pos': summary['precision_pos'],
        'recall_pos': summary['recall_pos'],
        'f1_pos': summary['f1_pos'],
        'cost': summary['cost'],
    })

threshold_results = pd.DataFrame(threshold_rows).sort_values(by='cost').reset_index(drop=True)
best_threshold = float(threshold_results.iloc[0]['threshold'])
save_dataframe(threshold_results, 'threshold_results.csv')
threshold_results.head(10)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(threshold_results['threshold'], threshold_results['cost'], marker='o')
plt.axvline(best_threshold, color='red', linestyle='--', label=f'Best threshold={best_threshold:.2f}')
plt.xlabel('Threshold for pos')
plt.ylabel('Cost')
plt.title('Cost Sensitivity to Decision Threshold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'cost_vs_threshold.png', dpi=300, bbox_inches='tight')
plt.show()

**Conclusión de la selección de umbral:** El umbral por defecto (`0.50`) no es el más conveniente cuando el objetivo es minimizar costo operativo. En la validación interna, un umbral de `0.15` reduce de forma importante el costo total y por eso se adopta como regla inicial de priorización.

## 9. Regla de decision final en validacion interna

In [ ]:
default_preds = np.where(catboost_scores >= 0.50, POSITIVE_LABEL, 'neg')
best_preds = np.where(catboost_scores >= best_threshold, POSITIVE_LABEL, 'neg')

default_summary = summarize_predictions(y_valid, default_preds)
best_summary = summarize_predictions(y_valid, best_preds)

threshold_comparison = pd.DataFrame([
    {
        'decision_rule': 'CatBoost threshold 0.50',
        'threshold': 0.50,
        'accuracy': default_summary['accuracy'],
        'f1_macro': default_summary['f1_macro'],
        'precision_pos': default_summary['precision_pos'],
        'recall_pos': default_summary['recall_pos'],
        'f1_pos': default_summary['f1_pos'],
        'cost': default_summary['cost'],
    },
    {
        'decision_rule': 'CatBoost best threshold',
        'threshold': best_threshold,
        'accuracy': best_summary['accuracy'],
        'f1_macro': best_summary['f1_macro'],
        'precision_pos': best_summary['precision_pos'],
        'recall_pos': best_summary['recall_pos'],
        'f1_pos': best_summary['f1_pos'],
        'cost': best_summary['cost'],
    },
])

save_dataframe(threshold_comparison, 'threshold_comparison_internal.csv')
threshold_comparison.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay(default_summary['cm'], display_labels=['neg', POSITIVE_LABEL]).plot(ax=axes[0], colorbar=False)
axes[0].set_title('CatBoost threshold 0.50')
ConfusionMatrixDisplay(best_summary['cm'], display_labels=['neg', POSITIVE_LABEL]).plot(ax=axes[1], colorbar=False)
axes[1].set_title(f'CatBoost best threshold ({best_threshold:.2f})')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'threshold_comparison_confusion.png', dpi=300, bbox_inches='tight')
plt.show()

**Conclusión de la regla de decisión interna:** Bajar el umbral incrementa el `recall` de la clase positiva y reduce el costo total, aunque sacrifica precisión. Esto es coherente con una estrategia de priorización, donde es preferible revisar más casos antes que dejar escapar fallas costosas.

## 10. Interpretabilidad basica

In [ ]:
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': catboost_model.get_feature_importance(),
}).sort_values(by='importance', ascending=False)

save_dataframe(feature_importance, 'feature_importance.csv')
feature_importance.head(15)

In [ ]:
top_features = feature_importance.head(15).sort_values(by='importance')
plt.figure(figsize=(8, 6))
plt.barh(top_features['feature'], top_features['importance'])
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Top 15 Feature Importances - CatBoost')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'top_feature_importances.png', dpi=300, bbox_inches='tight')
plt.show()

**Conclusión de la interpretabilidad:** El modelo no funciona como caja negra absoluta; es posible identificar variables con mayor peso en la predicción. En esta etapa, `ck_000` aparece como la variable más influyente, lo que abre una línea de interpretación de dominio para la discusión del informe.

## 11. Validacion con el conjunto de prueba oficial

In [ ]:
X_official_test = test_df.drop(columns=[TARGET_COLUMN]).copy()
y_official_test = test_df[TARGET_COLUMN].copy()

official_scores = catboost_model.predict_proba(X_official_test)[:, list(catboost_model.classes_).index(POSITIVE_LABEL)]
official_preds_default = np.where(official_scores >= 0.50, POSITIVE_LABEL, 'neg')
official_preds_best = np.where(official_scores >= best_threshold, POSITIVE_LABEL, 'neg')

official_default_summary = summarize_predictions(y_official_test, official_preds_default)
official_best_summary = summarize_predictions(y_official_test, official_preds_best)

official_test_results = pd.DataFrame([
    {
        'decision_rule': 'Official test - threshold 0.50',
        'threshold': 0.50,
        'accuracy': official_default_summary['accuracy'],
        'f1_macro': official_default_summary['f1_macro'],
        'precision_pos': official_default_summary['precision_pos'],
        'recall_pos': official_default_summary['recall_pos'],
        'f1_pos': official_default_summary['f1_pos'],
        'cost': official_default_summary['cost'],
    },
    {
        'decision_rule': 'Official test - best threshold',
        'threshold': best_threshold,
        'accuracy': official_best_summary['accuracy'],
        'f1_macro': official_best_summary['f1_macro'],
        'precision_pos': official_best_summary['precision_pos'],
        'recall_pos': official_best_summary['recall_pos'],
        'f1_pos': official_best_summary['f1_pos'],
        'cost': official_best_summary['cost'],
    },
])

save_dataframe(official_test_results, 'official_test_results.csv')
official_test_results.round(4)

**Conclusión de la validación oficial:** El modelo mantiene utilidad fuera de la partición interna. En el test oficial, el umbral `0.15` vuelve a reducir significativamente el costo total, lo que refuerza la idea de que la contribución del proyecto no es solo el modelo, sino también la calibración de la decisión.

## 12. Salida de priorizacion para aplicacion

In [ ]:
prioritization_table = test_df.copy().reset_index().rename(columns={'index': 'record_id'})
prioritization_table['failure_score'] = official_scores
prioritization_table['predicted_class'] = np.where(official_scores >= best_threshold, POSITIVE_LABEL, 'neg')

prioritization_table['priority_level'] = pd.cut(
    prioritization_table['failure_score'],
    bins=[-0.01, best_threshold, 0.50, 1.00],
    labels=['baja', 'media', 'alta'],
)

action_map = {
    'alta': 'inspeccion inmediata',
    'media': 'revision programada',
    'baja': 'seguimiento rutinario',
}
prioritization_table['recommended_action'] = prioritization_table['priority_level'].astype(str).map(action_map)

prioritization_output = prioritization_table[[
    'record_id',
    TARGET_COLUMN,
    'failure_score',
    'predicted_class',
    'priority_level',
    'recommended_action',
]]\
    .sort_values(by='failure_score', ascending=False)

save_dataframe(prioritization_output, 'prioritization_output.csv')
prioritization_output.head(15)

**Conclusión de la salida de priorización:** El proyecto ya no termina en una predicción binaria. A partir del score y el umbral optimizado, el sistema produce una tabla de prioridad y una acción sugerida, que puede alimentar una aplicación ligera o un dashboard de apoyo al mantenimiento.

## 13. Resumen ejecutivo del notebook

In [ ]:
final_notebook_summary = pd.DataFrame([
    {'item': 'selected_model', 'value': 'CatBoost'},
    {'item': 'best_threshold_by_cost', 'value': round(best_threshold, 2)},
    {'item': 'internal_cost_default_threshold', 'value': default_summary['cost']},
    {'item': 'internal_cost_best_threshold', 'value': best_summary['cost']},
    {'item': 'official_cost_default_threshold', 'value': official_default_summary['cost']},
    {'item': 'official_cost_best_threshold', 'value': official_best_summary['cost']},
    {'item': 'official_precision_pos_best_threshold', 'value': official_best_summary['precision_pos']},
    {'item': 'official_recall_pos_best_threshold', 'value': official_best_summary['recall_pos']},
    {'item': 'official_f1_pos_best_threshold', 'value': official_best_summary['f1_pos']},
    {'item': 'most_important_feature', 'value': feature_importance.iloc[0]['feature']},
])

save_dataframe(final_notebook_summary, 'final_notebook_summary.csv')
final_notebook_summary

**Conclusión final del notebook:** La base técnica del proyecto ya está construida. Se seleccionó `CatBoost` como modelo principal, se definió `0.15` como umbral inicial orientado a costo y se generó una salida de priorización reproducible. A partir de aquí, el siguiente paso natural del proyecto es construir la capa de aplicación ligera y avanzar en la redacción del informe académico.